[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Giocrisrai/mly1101-machine-learning/blob/main/notebooks/01_docente_solucionario.ipynb)

# MLY1101 · Machine Learning — Semana 01
## EA1 · Análisis y Preprocesamiento de Datos

**Resultado de aprendizaje (RA1):** implementar estrategias y técnicas de preprocesamiento
en el diseño de soluciones de Machine Learning, con un tratamiento responsable de la información.

---

### La idea central de hoy

Un proyecto de Machine Learning **no empieza eligiendo un algoritmo**. Empieza entendiendo
el problema y mirando los datos:

```
Problema → Datos → Exploración → Preprocesamiento → Modelamiento → Evaluación → Interpretación
           └────────── aquí estamos hoy ──────────┘
```

Hoy no vamos a entrenar ningún modelo. Vamos a hacer algo que decide el éxito o el fracaso
del modelo que entrenaremos más adelante: **entender y limpiar los datos**.

> Un modelo entrenado con datos que nadie revisó no es un modelo: es una opinión con decimales.

---

### El problema

Trabajas en el equipo de percepción de una empresa de conducción autónoma. El vehículo lleva
un sensor **LiDAR** que, varias veces por segundo, detecta objetos alrededor y entrega para
cada uno una *caja delimitadora* (bounding box) con su posición, tamaño y velocidad estimada.

El equipo de modelamiento quiere entrenar un clasificador que distinga **peatones, ciclistas,
vehículos y señalética**. Antes de gastar una sola hora en eso, alguien tiene que responder:

> **¿Podemos confiar en estas detecciones? ¿Qué tan sucios están los datos y qué habría que
> arreglar antes de modelar?**

Ese alguien eres tú, hoy.

---

### Sobre los datos

Esta actividad usa **Perception v2 real** (`datos/waymo_real/detecciones_reales.parquet`).
No hay un CSV de práctica en el repositorio: la licencia de Waymo prohíbe redistribuir los
datos, así que cada máquina los baja con `herramientas/descargar_waymo.py --muestra 40`.

El Open Dataset llega **curado** (0 % nulos, 0 valores imposibles). El trabajo de hoy no es
cazar suciedad plantada: es caracterizar lo que sí está — desbalance extremo de `cyclist`,
clima 100 % `sunny`, `LEVEL_2` raro, partir por `segment_id` — y decidir qué implica para
modelar.

---

### Al final de la sesión debes entregar

Un **mini-informe en Markdown** (última celda del notebook) con:

- 5 hallazgos sobre la calidad de los datos, cada uno respaldado con una cifra;
- 3 decisiones de preprocesamiento, cada una con su justificación;
- 1 riesgo ético o de sesgo identificado en el dataset.

> ### 🎓 Pauta docente
>
> **Cómo usar este documento.** Este es el solucionario del notebook
> `01_alumno_exploracion.ipynb`. Contiene el mismo contenido más: el código resuelto de cada
> TODO, las respuestas esperadas de cada pregunta de discusión (bloques `🎓 Pauta docente`) y
> los criterios de logro por bloque.
>
> **La actividad son 6 horas pedagógicas** según el programa. La distribución de abajo cubre
> el trabajo guiado; las horas restantes quedan para que apliquen lo mismo al caso oficial
> que hayan elegido (Telco, Housing o Spotify), sobre el que se rinde la Evaluación Parcial.
>
> **Distribución del bloque guiado (~4 h):**
>
> | Bloque | Min | Foco |
> |---|---|---|
> | 0 · El problema | 15 | Encuadre. Que nadie abra sklearn hoy. |
> | 1 · Carga e inspección | 45 | `.info()`, `.dtypes`, memoria, primer diagnóstico |
> | 2 · Tipos de variables | 45 | Taxonomía + categorías inconsistentes |
> | 3 · Nulos y duplicados | 45 | Nulos ocultos, patrón MNAR, duplicado lógico |
> | 4 · Outliers | 45 | IQR vs z, imposible vs legítimo |
> | 5 · Decisiones | 30 | Tabla de decisiones + fuga de información |
> | 6 · Datos responsables | 20 | Sesgo de muestreo, datos personales |
> | Cierre | 15 | Mini-informe |
>
> **El dataset es Perception v2 real** (medido 2026-09-08: 530.396 filas, 40 segmentos).
> Si un grupo dice "los datos están sucios", que muestre la cifra: v2 llega con 0 % nulos.
> Lo que sí hay que encontrar: `cyclist` 0,45 %, `weather` 100 % `sunny`, `LEVEL_2` 12,33 %,
> mediana `speed_mps` 0,0133.
>
> **Regla de oro de la clase:** ninguna afirmación sin una cifra que la respalde.

---
## Preparación del entorno

Ejecuta esta celda primero. Funciona tanto en Google Colab como en Jupyter local.

In [ ]:
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    REPO = Path("mly1101-machine-learning")
    if not REPO.exists():
        !git clone -q https://github.com/Giocrisrai/mly1101-machine-learning.git {REPO}
    RAIZ = REPO
else:
    # El notebook vive en notebooks/, así que la raíz del repositorio es la carpeta superior.
    RAIZ = Path("..").resolve()

sys.path.insert(0, str(RAIZ / "src"))
import waymo
RUTA_DATOS = waymo.exigir_detecciones_reales(RAIZ)

print("Colab:", EN_COLAB)
print("Raíz del repositorio:", RAIZ)
print("¿Existe el dataset?:", RUTA_DATOS.exists())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import eda  # utilidades de diagnóstico del repositorio: src/eda.py

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid")

print("pandas", pd.__version__, "| numpy", np.__version__)

---
# Bloque 1 · Carga e inspección inicial

**Preguntas que debemos responder antes de tocar nada:**

1. ¿Cuántas filas y columnas hay? ¿Cuánta memoria ocupan?
2. ¿Qué representa **una fila**? (esta es la pregunta más importante y la que más se salta)
3. ¿El tipo de dato que pandas infirió para cada columna es el que corresponde?

In [ ]:
df = pd.read_parquet(RUTA_DATOS)
print(f"Filas: {df.shape[0]:,}   Columnas: {df.shape[1]}")
df.head()

### 📖 Diccionario de datos

| Columna | Significado |
|---|---|
| `segment_id` | Identificador del segmento de conducción (~20 s de grabación) |
| `timestamp_micros` | Instante de la detección, en microsegundos |
| `id_interno` | Identificador único de la detección |
| `object_type` | Tipo de objeto detectado |
| `box_center_x/y/z` | Centro de la caja, en metros, respecto del vehículo (x = adelante) |
| `box_length/width/height` | Dimensiones de la caja, en metros |
| `speed_mps` | Velocidad estimada del objeto, en m/s |
| `num_lidar_points` | Cantidad de puntos láser que cayeron sobre el objeto |
| `weather` | Condición climática del segmento |
| `time_of_day` | Momento del día |
| `detection_difficulty` | Dificultad de la detección según el sensor (LEVEL_1 = fácil) |
| `location` | Ciudad del segmento (`location_sf` / `location_phx`) |

**Una fila = una detección de un objeto en un instante determinado.** No es un objeto, ni un
segmento, ni un vehículo. Ténlo presente: define qué significa "duplicado" más adelante.

### ✏️ TODO 1

Obtén, en una sola celda:

1. la estructura del DataFrame con `.info()`;
2. el uso de memoria **real** (`memory_usage(deep=True)`) en MB.

In [ ]:
df.info()
memoria_mb = df.memory_usage(deep=True).sum() / 1024**2
print(f"\nMemoria real: {memoria_mb:.1f} MB")

### ✏️ TODO 2 — El primer problema

Mira la salida anterior con atención. Hay una columna cuyo tipo **no es el que debería ser**.

1. Identifica el dtype de `timestamp_micros` en el parquet.
2. Convierte con `pd.to_numeric(..., errors="coerce")` y cuenta cuántos valores **no** son
   numéricos. En v2 el resultado es la lección: Parquet conservó el tipo; un CSV habría
   podido ensuciar toda la columna con un solo `"N/D"`.

In [ ]:
# timestamp_micros en parquet real ya es entero. ¿Cuántos valores no son numéricos?
convertidos = pd.to_numeric(df["timestamp_micros"], errors="coerce")
no_convertibles = df.loc[convertidos.isna(), "timestamp_micros"]

print("dtype actual:", df["timestamp_micros"].dtype)
print("Valores no convertibles a número:", len(no_convertibles))
print(no_convertibles.value_counts())

In [ ]:
# Autochequeo — Perception v2 conserva tipos: el timestamp ya es int64.
assert pd.api.types.is_integer_dtype(df["timestamp_micros"]), (
    "revisa: en el parquet real el timestamp debe ser entero, no texto"
)
assert len(no_convertibles) == 0, "v2 no trae 'N/D': si aparecen, no es el parquet del curso"
print(f"✅ Hallazgo 1: timestamp_micros es {df['timestamp_micros'].dtype}, 0 valores no numéricos.")
print("   (Si esto fuera un CSV, un solo texto ensuciaría toda la columna. Parquet no lo permite.)")

> ### 🎓 Pauta docente — TODO 2
>
> **Respuesta:** `timestamp_micros` en el parquet **ya es `int64`**. `to_numeric` no convierte
> nada: 0 valores no numéricos. Parquet conservó el tipo. Un CSV habría podido ensuciar toda
> la columna con un solo `"N/D"`.
>
> **Pregunta para el curso:** ¿por qué igual convertimos? Porque el hábito de auditar el dtype
> se lleva a cualquier fuente. Aquí el hallazgo es *cero suciedad de tipo*; eso también se
> documenta.
>
> **Error frecuente:** el alumno hace `df.dropna()` esperando que desaparezca un `"N/D"`. No
> hay ninguno. `"N/D"` **no es** un nulo para pandas: si apareciera, `dropna()` no lo vería.
>
> **Criterio de logro:** identifica el dtype, cuantifica (0 no numéricos) y explica por qué
> el formato importa.

### Diagnóstico general

En vez de revisar columna por columna a mano, usamos `eda.resumen_calidad()`, que entrega una
radiografía completa: tipo, cardinalidad, nulos y **valores centinela** (valores que
representan un dato faltante sin ser `NaN`, como `-1` o `"N/D"`).

In [ ]:
resumen = eda.resumen_calidad(df)
resumen

### ✏️ TODO 3

Usando la tabla anterior, responde en la celda de texto de abajo:

1. ¿Qué columna tiene **cardinalidad casi 100 %**? ¿Sirve como variable predictora? ¿Por qué?
2. ¿Qué columna es **constante**? ¿Qué aporta a un modelo?
3. ¿Qué columnas tienen nulos declarados (`NaN`) y cuáles tienen **nulos ocultos** (centinelas)?

In [ ]:
print("Columnas de cardinalidad casi única (no son features):")
print(resumen[resumen["pct_unicos"] > 90][["dtype", "n_unicos", "pct_unicos"]], "\n")

print("Columnas constantes (no aportan información):")
print(resumen[resumen["n_unicos"] <= 1][["dtype", "n_unicos", "ejemplos"]], "\n")

print("Columnas con algo faltante (declarado u oculto):")
print(resumen[resumen["pct_faltante_total"] > 0][["n_nulos", "n_centinelas", "pct_faltante_total"]])

**✍️ Tu respuesta al TODO 3:**

*(doble clic aquí y escribe)*

1.
2.
3.

> ### 🎓 Pauta docente — TODO 3
>
> 1. **`id_interno`** tiene ~98 % de valores únicos. No sirve como predictor: un identificador
>    no tiene relación causal con nada. Si se lo damos a un modelo con capacidad suficiente,
>    memoriza el identificador y el rendimiento en test se desploma. Sirve como llave, no como
>    feature. (`segment_id` es distinto: agrupa detecciones y **sí** sirve para razonar sobre
>    dependencia entre filas y para armar el split más adelante.)
> 2. **`weather`** es constante en este lote (`sunny`, 100 %). Varianza cero ⇒ no discrimina
>    detecciones, pero **sí es un hallazgo de cobertura**: el sistema no va a operar solo con
>    sol. No se tira a la ligera: se documenta. (`location` sí varía: SF / Phoenix.)
> 3. Nulos declarados: **0**. Nulos ocultos (`-1`, `"N/D"`): **0**. v2 llega curado. El trabajo
>    no es cazar 10 defectos plantados: es el desbalance (`cyclist` 0,45 %, `LEVEL_2` 12,33 %)
>    y el sesgo de muestreo.
>
> **Punto clave del bloque:** `df.isna().sum()` **no** basta para auditar los faltantes.

---
# Bloque 2 · Tipos de variables y categorías

El tipo que usa pandas (`int64`, `object`, …) no es lo mismo que el **tipo estadístico** de la
variable, y es el tipo estadístico el que decide qué se puede hacer con ella:

| Tipo estadístico | Definición | Ejemplo aquí | ¿Media? |
|---|---|---|---|
| **Nominal** | categorías sin orden | `object_type`, `weather` | ❌ |
| **Ordinal** | categorías con orden | `detection_difficulty` | ❌ (sí mediana) |
| **Discreta** | numérica, se cuenta | `num_lidar_points` | ✅ |
| **Continua** | numérica, se mide | `speed_mps`, `box_length` | ✅ |

`timestamp_micros` es un caso aparte: es numérica, pero su significado es **temporal**. Calcular
su promedio no tiene sentido; calcular diferencias, sí.

### ✏️ TODO 4

Completa el diccionario clasificando cada columna. Después ejecuta el autochequeo.

In [ ]:
tipos_estadisticos = {
    "segment_id": "nominal",
    "timestamp_micros": "temporal",
    "id_interno": "identificador",
    "object_type": "nominal",
    "box_center_x": "continua",
    "box_center_y": "continua",
    "box_center_z": "continua",
    "box_length": "continua",
    "box_width": "continua",
    "box_height": "continua",
    "speed_mps": "continua",
    "num_lidar_points": "discreta",
    "weather": "nominal",
    "time_of_day": "nominal",
    "detection_difficulty": "ordinal",
    "location": "nominal",
}

In [ ]:
# Autochequeo
faltantes = set(df.columns) - set(tipos_estadisticos)
assert not faltantes, f"faltan columnas por clasificar: {faltantes}"
assert "____" not in tipos_estadisticos.values(), "quedaron casilleros sin completar"
assert tipos_estadisticos["num_lidar_points"] == "discreta", "se cuentan puntos: es discreta"
assert tipos_estadisticos["detection_difficulty"] == "ordinal", "LEVEL_1 < LEVEL_2: hay orden"
print("✅ Clasificación completa y coherente.")

### El problema de las categorías

Ahora miremos qué categorías existen realmente en las variables nominales. Aquí es donde
aparecen los problemas que ningún `.info()` muestra.

### ✏️ TODO 5

Muestra la frecuencia de cada valor de `object_type` y de `weather`, **incluyendo los nulos**.

In [ ]:
print(df["object_type"].value_counts(dropna=False), "\n")
print(df["weather"].value_counts(dropna=False))

Cuenta las categorías que ves. ¿Cuántos tipos de objeto hay **en realidad**? ¿Cuántas condiciones
climáticas distintas existen **en realidad**?

### ✏️ TODO 6

Normaliza ambas columnas: quita espacios, unifica mayúsculas y traduce las variantes a una
forma canónica. Usa `eda.normalizar_categoria(serie, mapa)`.

*Ojo con `"RAIN "` y `" rain"`: los espacios son invisibles en pantalla pero `pandas` los cuenta
como categorías distintas.*

In [ ]:
mapa_objetos = {"peaton": "pedestrian", "ped": "pedestrian"}
mapa_clima = {"soleado": "sunny", "lluvia": "rain", "niebla": "fog"}

df["object_type_limpio"] = eda.normalizar_categoria(df["object_type"], mapa_objetos)
df["weather_limpio"] = eda.normalizar_categoria(df["weather"], mapa_clima)

print(df["object_type_limpio"].value_counts(dropna=False), "\n")
print(df["weather_limpio"].value_counts(dropna=False))

In [ ]:
# Autochequeo
assert set(df["object_type_limpio"].dropna().unique()) == {"vehicle", "pedestrian", "cyclist", "sign"}, \
    "deben quedar exactamente 4 tipos de objeto"
assert set(df["weather_limpio"].dropna().unique()) == {"sunny"}, \
    "en este lote v2 el clima es 100 % sunny; si ves rain/fog, no es el parquet del curso"
print("✅ object_type ya viene unificado (4 tipos). weather: solo sunny — sesgo de muestreo, no suciedad.")

> ### 🎓 Pauta docente — TODOs 5 y 6
>
> **Lo que debe pasar:** el alumno cuenta 7 valores distintos en `object_type` y 11 en `weather`,
> y descubre que en realidad son 4 y 3.
>
> **La pregunta que hay que hacer:** *"Si el equipo de modelamiento entrena con esto tal cual,
> ¿qué pasa?"*
> Respuesta: un one-hot encoding genera 7 columnas para 4 clases reales. El modelo trata
> `PEATON` y `Pedestrian` como cosas distintas, reparte la evidencia entre categorías gemelas y
> aprende peor de cada una. Y si en producción llega `"Peaton"` (una variante nueva), no
> corresponde a ninguna columna aprendida.
>
> **El detalle de los espacios** vale la pena mostrarlo en vivo:
> `df["weather"].unique()` muestra `'RAIN '` y `'rain'` casi idénticos en pantalla. Sugerencia:
> ejecutar `[repr(v) for v in df["weather"].dropna().unique()]` para que los espacios se vean.
>
> **Advertencia metodológica:** normalizamos en columnas *nuevas* (`_limpio`) en vez de
> sobreescribir. Así el dataset original queda auditable y podemos comparar antes/después. Es
> una buena práctica que conviene explicitar.
>
> **Criterio de logro:** deja las 4 y 3 categorías correctas y explica el impacto en el modelo.

### Desbalance de clases

Con las categorías ya limpias, podemos ver algo que antes estaba oculto: cómo se reparten
las clases que el equipo quiere predecir.

In [ ]:
desbalance = eda.resumen_desbalance(df["object_type_limpio"])
print(desbalance)

fig, ax = plt.subplots(figsize=(7, 3.5))
desbalance["pct"].sort_values().plot.barh(ax=ax, color="#4C72B0")
ax.set_xlabel("% de detecciones")
ax.set_ylabel("")
ax.set_title("Composición del dataset por tipo de objeto")
for i, valor in enumerate(desbalance["pct"].sort_values()):
    ax.text(valor + 0.7, i, f"{valor:.1f}%", va="center")
plt.tight_layout()
plt.show()

> ### 🎓 Pauta docente — desbalance
>
> `CYCLIST` es ~2 % de las filas: hay ~30 vehículos por cada ciclista.
>
> **Preguntas para el curso:**
> - *"Si mi modelo predice siempre VEHICLE, ¿qué exactitud obtiene?"* → ~62 %. Y es inútil.
>   Aquí queda sembrada la discusión de la Actividad 2.2 sobre por qué el *accuracy* engaña.
> - *"¿Cuál es la clase donde equivocarse cuesta vidas?"* → ciclista y peatón. Justamente las
>   más difíciles de detectar y, en el caso del ciclista, la más escasa. El desbalance no es un
>   problema estadístico abstracto: es un problema de seguridad.
>
> **No corresponde todavía** hablar de SMOTE ni de `class_weight`. Basta con dejar registrado el
> hallazgo. Se retoma en la Actividad 2.2.

---
# Bloque 3 · Datos faltantes y duplicados

Tres preguntas, en este orden:

1. ¿Cuántos faltan? (lo fácil)
2. ¿Están **escondidos** detrás de un valor válido? (lo que casi nadie revisa)
3. ¿Faltan **al azar** o siguen un patrón? (lo que decide qué podemos hacer con ellos)

### ✏️ TODO 7

Calcula, para cada columna, el número y el porcentaje de nulos declarados, mostrando solo las
columnas que tengan al menos uno.

In [ ]:
nulos = pd.DataFrame({
    "n_nulos": df.isna().sum(),
    "pct": (100 * df.isna().mean()).round(2),
})
nulos[nulos["n_nulos"] > 0].sort_values("n_nulos", ascending=False)

### Nulos ocultos

`isna()` solo ve lo que pandas reconoce como faltante. Un dato faltante también puede estar
disfrazado de valor válido: `-1`, `0`, `-999`, `"N/D"`, `"sin dato"`.

### ✏️ TODO 8

`num_lidar_points` es un conteo de puntos láser. Por definición **no puede ser negativo**.
Averigua cuántas filas violan esa regla y qué valor usan.

In [ ]:
print(df["num_lidar_points"].describe(), "\n")
n_centinela = (df["num_lidar_points"] == -1).sum()
print(f"Filas con -1: {n_centinela:,} ({100 * n_centinela / len(df):.2f}%)")
print("Nulos que pandas ve en esa columna:", df["num_lidar_points"].isna().sum())

### ✏️ TODO 9

Crea las versiones corregidas de las dos columnas contaminadas, convirtiendo el valor centinela
en un `NaN` explícito:

- `num_lidar_points_limpio`: igual que la original, pero con `-1` → `NaN`.
- `timestamp_limpio`: la marca de tiempo convertida a número, con `"N/D"` → `NaN`.

In [ ]:
df["num_lidar_points_limpio"] = df["num_lidar_points"].replace(-1, np.nan)
df["timestamp_limpio"] = eda.a_numerico(df["timestamp_micros"])

print(df[["num_lidar_points_limpio", "timestamp_limpio"]].isna().sum())
print("\nTipos:", df["num_lidar_points_limpio"].dtype, "|", df["timestamp_limpio"].dtype)

In [ ]:
# Autochequeo — v2 no disfraza faltantes con -1 ni con "N/D".
assert df["num_lidar_points_limpio"].isna().sum() == 0, "v2 no trae centinela -1"
assert (df["num_lidar_points_limpio"] >= 0).all(), "no pueden quedar conteos negativos"
assert pd.api.types.is_numeric_dtype(df["timestamp_limpio"]), "el timestamp debe ser numérico"
print("✅ 0 nulos ocultos. El Open Dataset llega curado; el trabajo está en el desbalance, no en el sucio.")

### ¿Los nulos son aleatorios?

Esta es **la** pregunta del bloque. Tres escenarios posibles:

| Mecanismo | Significa | Consecuencia |
|---|---|---|
| **MCAR** | falta al azar puro | eliminar filas es (casi) inofensivo |
| **MAR** | la falta depende de *otras* variables observadas | se puede imputar condicionando |
| **MNAR** | la falta depende del *propio* valor faltante | eliminar **sesga** el dataset |

Veamos el caso de `speed_mps`.

### ✏️ TODO 10

Cruza el porcentaje de nulos de `speed_mps` por `detection_difficulty` y `time_of_day`. Usa
`eda.matriz_nulos_por_grupo(df, columna, [grupo1, grupo2])`.

In [ ]:
patron = eda.matriz_nulos_por_grupo(df, "speed_mps", ["detection_difficulty", "time_of_day"])
print(patron, "\n")

fig, ax = plt.subplots(figsize=(6.5, 3))
sns.heatmap(patron, annot=True, fmt=".1f", cmap="Reds", cbar_kws={"label": "% nulos"}, ax=ax)
ax.set_title("% de velocidad faltante según dificultad y momento del día")
plt.tight_layout()
plt.show()

**✍️ Discusión (escribe tu respuesta):**

Si el equipo decide `df.dropna(subset=["speed_mps"])` antes de entrenar:

1. ¿Qué tipo de detecciones desaparecen del dataset?
2. ¿Qué le pasa al modelo entrenado con lo que queda cuando el auto circula **de noche**?
3. ¿Es esto MCAR, MAR o MNAR?

*(doble clic y responde)*

> ### 🎓 Pauta docente — TODO 10 (el momento más importante de la clase)
>
> **Las cifras:** en `LEVEL_1` falta ~0,4 % de las velocidades sin importar la hora. En
> `LEVEL_2` de noche falta ~**34 %**. No es azar: es el sensor fallando justo cuando le cuesta.
>
> **Respuestas esperadas:**
> 1. Desaparecen casi solo las detecciones **difíciles y nocturnas**.
> 2. El modelo se entrena con un mundo más fácil y más iluminado que el real. Su desempeño
>    medido en validación será optimista y su desempeño real de noche, peor de lo esperado.
>    Y de noche es cuando más importa.
> 3. **MNAR** (o MAR según cómo se argumente, y ambas defensas son válidas si están
>    fundamentadas): la falta depende de condiciones que también afectan al valor mismo. Lo que
>    NO es, es MCAR, y eso es lo que hay que dejar claro.
>
> **La frase para cerrar el bloque:** *"Eliminar filas nunca es gratis. Siempre estás eligiendo
> qué parte de la realidad borrar."*
>
> **Extensión si sobra tiempo:** ¿qué haría el alumno en vez de eliminar? Opciones razonables:
> imputar por mediana **dentro de cada grupo** (tipo de objeto × dificultad), o agregar una
> columna indicadora `speed_faltante` que le diga al modelo que ahí no había medición. Esta
> última suele ser la mejor y casi nadie la propone sola.
>
> **Criterio de logro:** describe el patrón con cifras y conecta el faltante con un riesgo
> concreto de seguridad, no solo con "el modelo pierde datos".

### Duplicados

Recuerda: **una fila = una detección de un objeto en un instante**. Entonces, dos filas con el
mismo `id_interno` son, por definición, un error.

Hay dos tipos de duplicado y solo uno se resuelve con `drop_duplicates()`:

- **Duplicado exacto:** la fila completa está repetida.
- **Duplicado lógico:** se repite la *llave*, pero los demás valores difieren. `drop_duplicates()`
  no lo detecta, porque para pandas las filas son distintas.

### ✏️ TODO 11

Cuantifica ambos tipos usando `eda.reporte_duplicados(df, llave)` con la llave
`["segment_id", "timestamp_micros", "id_interno"]`, y muestra un ejemplo concreto de duplicado
lógico.

In [ ]:
LLAVE = ["segment_id", "timestamp_micros", "id_interno"]
print(eda.reporte_duplicados(df, LLAVE), "\n")

# Un ejemplo concreto: una llave repetida cuyas filas NO son idénticas.
repetidas = df[df.duplicated(subset=LLAVE, keep=False)]
for _, grupo in repetidas.groupby(LLAVE):
    if grupo.drop_duplicates().shape[0] > 1:
        display(grupo[LLAVE + ["object_type", "box_center_x", "num_lidar_points"]])
        break

In [ ]:
# Autochequeo — v2 no trae duplicados plantados.
reporte = eda.reporte_duplicados(df, LLAVE).iloc[0]
assert reporte["dup_exactos"] == 0, "v2 no debería traer duplicados exactos"
assert reporte["dup_logicos"] == 0, "ni duplicados lógicos"
print("✅ 0 duplicados exactos y 0 lógicos. drop_duplicates() aquí no cambia nada.")

> ### 🎓 Pauta docente — TODO 11
>
> **Cifras esperadas (Perception v2, 530.396 filas):** **0** duplicados exactos y **0**
> duplicados lógicos. `drop_duplicates()` aquí no cambia nada.
>
> **La demostración que conviene hacer en vivo:** ejecutar el reporte y mostrar el cero. El
> método es el mismo que usarías si el cero no estuviera: en un pipeline real los duplicados
> aparecen. Aquí el hallazgo es *curado*, y también se documenta.
>
> **Preguntas:**
> - *"¿Por qué un sistema real sí tiene duplicados?"* → reprocesamiento, mezcla de exportaciones,
>   reintentos. Es lo normal en un pipeline, no una rareza.
> - *"¿Con cuál de las dos filas te quedas si aparecen?"* → hay que **decidir y documentar**.
>
> **Criterio de logro:** distingue ambos tipos, los cuantifica (aquí: cero) y propone una regla
> de desempate por si mañana no es cero.

---
# Bloque 4 · Valores atípicos

Un valor atípico puede ser dos cosas muy distintas:

- un **error de medición** (el sensor falló) → hay que corregirlo o eliminarlo;
- un **caso real poco frecuente** (existe un bus) → eliminarlo es destruir información valiosa.

Los métodos estadísticos **no distinguen entre ambos**. Esa distinción la hace quien conoce el
dominio. Por eso este bloque no se trata de aplicar una fórmula, sino de mirar los datos.

In [ ]:
numericas = ["box_center_x", "box_center_y", "box_center_z",
             "box_length", "box_width", "box_height", "speed_mps"]
eda.perfil_numerico(df, numericas)

### ✏️ TODO 12

Mira la fila de `speed_mps` en la tabla anterior: compara la mediana (`50%`) con el máximo.

Grafica la distribución de `speed_mps` con un histograma y un boxplot, y responde: ¿es plausible
el máximo? (1 m/s = 3,6 km/h).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
df["speed_mps"].plot.hist(bins=80, ax=axes[0], color="#4C72B0")
axes[0].set_title("Distribución de speed_mps")
axes[0].set_xlabel("m/s")

sns.boxplot(x=df["speed_mps"], ax=axes[1])
axes[1].set_title("Boxplot de speed_mps")
plt.tight_layout()
plt.show()

maximo = df["speed_mps"].max()
print(f"Máximo observado: {maximo:.1f} m/s = {maximo * 3.6:.0f} km/h")
print(f"Detecciones sobre 60 m/s (216 km/h): {(df['speed_mps'] > 60).sum()}")

### El criterio del rango intercuartil (IQR)

Se marca como atípico todo valor fuera del intervalo

$$[\,Q_1 - k\cdot IQR,\;\; Q_3 + k\cdot IQR\,], \qquad IQR = Q_3 - Q_1$$

con $k = 1{,}5$ para atípicos moderados y $k = 3$ para extremos.

Una alternativa es el **puntaje z**: $z = (x - \mu)/\sigma$, atípico si $|z| > 3$. Pero $\mu$ y
$\sigma$ se calculan *con* los outliers incluidos, así que un valor extremo infla $\sigma$ y se
esconde a sí mismo. El IQR, basado en cuantiles, es más robusto.

### ✏️ TODO 13

Compara ambos criterios sobre `speed_mps`: ¿cuántos valores marca cada uno?

In [ ]:
por_iqr = eda.detectar_outliers_iqr(df["speed_mps"], k=1.5)
por_z = eda.detectar_outliers_zscore(df["speed_mps"], umbral=3)
inferior, superior = eda.limites_iqr(df["speed_mps"], k=1.5)

print(f"Límites IQR: [{inferior:.2f}, {superior:.2f}] m/s")
print(f"Marcados por IQR:     {por_iqr.sum():>5}")
print(f"Marcados por z-score: {por_z.sum():>5}")
print(f"Marcados por ambos:   {(por_iqr & por_z).sum():>5}")

> ### 🎓 Pauta docente — TODOs 12 y 13
>
> **Cifras:** el máximo de `speed_mps` ronda los 338 m/s ≈ **1.218 km/h**. Un peatón a velocidad
> de avión comercial. ~160 detecciones superan los 60 m/s.
>
> **Punto clave del bloque:** el IQR marca **muchos más** valores que el z-score. Y aquí viene la
> trampa: el IQR marca a los buses (grandes pero reales) y el z-score deja pasar velocidades
> imposibles, porque esos mismos valores inflaron σ. **Ningún criterio automático sabe cuál es
> cuál.**
>
> Vale la pena escribirlo en la pizarra: *el umbral estadístico propone, el conocimiento del
> dominio dispone*.

### ✏️ TODO 14 — Atípico imposible vs. atípico legítimo

Ahora la parte que ninguna fórmula resuelve. Revisa los valores atípicos de `box_length`:

1. ¿Cuántos son **negativos**? ¿Puede existir un objeto de largo negativo?
2. Los objetos con largo mayor a 12 m, ¿son errores? Mira su ancho, su alto y su tipo antes de
   responder.

In [ ]:
atipicos_largo = eda.detectar_outliers_iqr(df["box_length"])
print(f"Atípicos de box_length según IQR: {atipicos_largo.sum()}\n")

print("--- Largo negativo (imposible) ---")
print(f"{(df['box_length'] < 0).sum()} filas\n")

print("--- Largo > 12 m: ¿error o realidad? ---")
grandes = df[df["box_length"] > 12]
print(grandes[["object_type_limpio", "box_length", "box_width", "box_height"]].describe().round(2))
print("\nTipos de objeto involucrados:", grandes["object_type_limpio"].unique())

**✍️ Tu conclusión:** ¿qué harías con cada uno de los dos grupos y por qué?

*(doble clic y responde)*

> ### 🎓 Pauta docente — TODO 14
>
> **Los negativos (~80 filas):** físicamente imposibles. Es una falla del sensor o del
> exportador. Se tratan como faltantes (`NaN`), no se "arreglan" tomando valor absoluto: no
> sabemos si el largo real era ese número.
>
> **Los mayores a 12 m (~600 filas):** son todos `VEHICLE`, con ancho ~2,6 m y alto ~3,2 m.
> Es decir: **buses y camiones**. Son reales, son exactamente el tipo de objeto que un auto
> autónomo no puede permitirse ignorar, y el IQR los marcó como atípicos.
>
> **La pregunta decisiva para el curso:** *"Si eliminamos todos los atípicos de `box_length`,
> ¿qué acabamos de hacer?"* → Entrenar un auto autónomo que nunca vio un bus.
>
> **Regla práctica que conviene dejar escrita:**
> 1. Definir primero las reglas de dominio (qué es físicamente imposible).
> 2. Tratar lo imposible como dato faltante.
> 3. Lo raro pero posible **se conserva**, y se documenta.
>
> **Criterio de logro:** el alumno propone tratamientos **distintos** para ambos grupos y
> justifica el porqué con el dominio, no con estadística.

### ✏️ TODO 15 — Reglas de dominio

En lugar de confiar en un umbral estadístico, escribamos explícitamente **qué es imposible**
en este dominio. Completa el diccionario de reglas: cada valor es una expresión que describe
las filas **inválidas**.

In [ ]:
reglas = {
    "largo no positivo": "box_length <= 0",
    "alto no positivo": "box_height <= 0",
    "ancho no positivo": "box_width <= 0",
    "velocidad sobre 60 m/s (216 km/h)": "speed_mps > 60",
    "conteo de puntos negativo": "num_lidar_points < 0",
    "peatón más alto que 2.5 m": "object_type_limpio == 'pedestrian' and box_height > 2.5",
}
eda.valores_imposibles(df, reglas)

---
# Bloque 5 · De los hallazgos a las decisiones

Encontrar problemas es la mitad del trabajo. La otra mitad es **decidir qué hacer con cada uno
y dejarlo documentado**, porque cada decisión cambia los datos con los que se entrenará el
modelo.

### ✏️ TODO 16

Completa esta tabla con tus decisiones. Es el corazón de tu entrega.

| Columna | Problema detectado | Cifra | Decisión | Justificación |
|---|---|---|---|---|
| `timestamp_micros` | ¿dtype texto / `"N/D"`? | | | |
| `num_lidar_points` | ¿`-1` centinela? | | | |
| `weather` | cobertura (¿constante?) | | | |
| `object_type` | desbalance | | | |
| duplicados | exactos y lógicos | | | |
| `box_length` | imposibles vs atípicos | | | |
| `speed_mps` | nulos / imposibles | | | |
| `location` | sesgo geográfico | | | |

*(doble clic para editar la tabla)*

> ### 🎓 Pauta docente — TODO 16 (tabla de referencia)
>
> | Columna | Problema | Cifra | Decisión razonable | Justificación |
> |---|---|---|---|---|
> | `timestamp_micros` | dtype (¿texto?) | `int64`, 0 `"N/D"` | Dejar; documentar que Parquet conservó el tipo | Un CSV habría roto la columna |
> | `num_lidar_points` | ¿centinela `-1`? | 0 | El código igual convierte centinelas | Hábitos de auditoría |
> | `weather` | constante `sunny` | **100 %** | Conservar y documentar el sesgo | Cobertura, no suciedad |
> | `object_type` | desbalance | `cyclist` **0,45 %** | No tirar la minoría | Ética + modelo |
> | duplicados | exactos / lógicos | **0 / 0** | Regla lista por si aparecen | Auditar no es inventar suciedad |
> | `box_length` | imposibles vs buses | 0 imposibles; max > 12 m | Conservar atípicos reales | Solo lo imposible es error |
> | `speed_mps` | nulos / imposibles | 0 nulos; max 34 m/s | No imputar lo que no falta | `dropna()` global no aplica |
> | `location` | sesgo geográfico | SF 75 % · PHX 25 % | Documentar; no es feature de la caja | El sistema no opera solo en 2 ciudades |
>
> Se acepta cualquier decisión distinta **si está justificada**. Lo que no se acepta es
> `df.dropna()` sin argumento.

### ✏️ TODO 17 — Aplica tus decisiones

Escribe una función que reciba el DataFrame crudo y devuelva el limpio. Que sea una función y no
celdas sueltas importa: es reproducible, se puede testear y se puede volver a aplicar a datos
nuevos.

In [ ]:
def limpiar(datos: pd.DataFrame) -> pd.DataFrame:
    """Aplica las decisiones de preprocesamiento acordadas y devuelve una copia limpia."""
    d = datos.copy()

    # 1. Tipos y valores centinela
    d["timestamp_micros"] = eda.a_numerico(d["timestamp_micros"])
    d["num_lidar_points"] = d["num_lidar_points"].replace(-1, np.nan)

    # 2. Categorías
    d["object_type"] = eda.normalizar_categoria(d["object_type"], {"peaton": "pedestrian", "ped": "pedestrian"})
    d["weather"] = eda.normalizar_categoria(d["weather"], {"soleado": "sunny", "lluvia": "rain", "niebla": "fog"})
    d["weather"] = d["weather"].fillna("desconocido")

    # 3. Valores físicamente imposibles -> faltantes (NO se corrigen: no sabemos el valor real)
    for columna in ["box_length", "box_width", "box_height"]:
        d.loc[d[columna] <= 0, columna] = np.nan
    d.loc[d["speed_mps"] > 60, "speed_mps"] = np.nan

    # 4. Indicador de faltante: le dice al modelo que ahí no hubo medición
    d["speed_faltante"] = d["speed_mps"].isna().astype(int)

    # 5. Duplicados: primero los exactos, después los lógicos con una regla explícita
    #    (nos quedamos con la detección que tiene más puntos láser: es la mejor medición).
    d = d.drop_duplicates()
    d = (d.sort_values("num_lidar_points", ascending=False, na_position="last")
           .drop_duplicates(subset=["segment_id", "timestamp_micros", "id_interno"], keep="first")
           .sort_index())

    # 6. Columnas que no existen en v2 (p. ej. sensor_version del hilo viejo)
    d = d.drop(columns=["sensor_version"], errors="ignore")

    return d


df_limpio = limpiar(pd.read_parquet(RUTA_DATOS))
print(f"Crudo:  {len(df):,} filas")
print(f"Limpio: {len(df_limpio):,} filas  ({len(df) - len(df_limpio):,} eliminadas)")
df_limpio.head(3)

In [ ]:
# Autochequeo del dataset limpio
assert df_limpio.duplicated(subset=["segment_id", "timestamp_micros", "id_interno"]).sum() == 0, \
    "no deben quedar llaves repetidas"
assert set(df_limpio["object_type"].unique()) == {"vehicle", "pedestrian", "cyclist", "sign"}
assert "sensor_version" not in df_limpio.columns
assert (df_limpio["box_length"].dropna() > 0).all(), "no deben quedar largos imposibles"
assert df_limpio["box_length"].max() > 12, "los buses deben SEGUIR AHÍ: no son errores"
assert pd.api.types.is_numeric_dtype(df_limpio["timestamp_micros"])
print("✅ Dataset limpio y auditable. Los casos raros pero reales siguen presentes.")

### ⚠️ Una advertencia para las próximas semanas: la fuga de información

Fíjate en algo que **no** hicimos: no imputamos los nulos con la media ni escalamos ninguna
variable.

No es un olvido. Si calculas la media de todo el dataset y con ella rellenas los nulos, y
**después** separas entrenamiento y prueba, el conjunto de prueba ya influyó en el conjunto de
entrenamiento a través de esa media. Eso se llama **fuga de información** (*data leakage*), y su
síntoma es un modelo que rinde excelente en las pruebas y mal en producción.

El orden correcto es:

```
limpieza estructural (lo de hoy)  →  separar train/test  →  ajustar imputación y escalado SOLO con train  →  aplicar a test
```

En la Actividad 2.2 haremos esto con `Pipeline` y `ColumnTransformer` de scikit-learn, que existen justamente
para que este error sea difícil de cometer.

---
# Bloque 6 · Tratamiento responsable de la información

Los datos de conducción autónoma se recogen **en la vía pública**, donde hay personas que nunca
dieron su consentimiento. Antes de modelar, tres preguntas:

1. **¿Hay datos personales aquí?** El dataset no tiene nombres ni rostros, pero sí posiciones de
   peatones asociadas a un instante y un segmento. Si el segmento tiene geolocalización (los
   datos reales de Waymo la tienen), la combinación *lugar + hora + trayectoria* puede
   reidentificar a una persona. Anonimizar no es solo borrar la columna "nombre".
2. **¿Cómo se recolectó?** En los datos reales, con cámaras y LiDAR en vía pública. Waymo difumina
   rostros y patentes antes de publicar. Esa decisión es parte del diseño del dataset, no un
   detalle técnico.
3. **¿A quién representa mal este dataset?** Es la pregunta del ejercicio siguiente.

### ✏️ TODO 18

Calcula la composición del dataset por clima y momento del día, y el porcentaje de detecciones
que ocurren de noche o con lluvia.

In [ ]:
composicion = pd.crosstab(df_limpio["weather"], df_limpio["time_of_day"], normalize="all") * 100
print(composicion.round(2), "\n")

pct_noche = 100 * (df_limpio["time_of_day"] == "Night").mean()
pct_lluvia = 100 * (df_limpio["weather"] == "rain").mean()
pct_dificil_noche = 100 * ((df_limpio["time_of_day"] == "Night") &
                           (df_limpio["detection_difficulty"] == "LEVEL_2")).mean()

print(f"Detecciones nocturnas: {pct_noche:.1f}%")
print(f"Detecciones con lluvia: {pct_lluvia:.1f}%")
print(f"Nocturnas Y difíciles: {pct_dificil_noche:.1f}%")
print(f"Ciclistas: {100 * (df_limpio['object_type'] == 'cyclist').mean():.1f}%")

**✍️ Discusión final (escribe tu respuesta):**

Un modelo entrenado con este dataset se instalará en autos que circulan **de noche y con lluvia**,
y que se cruzan con **ciclistas**.

1. ¿Qué situación está sub-representada en los datos?
2. ¿Qué grupo de personas corre más riesgo si el modelo falla en esa situación?
3. Nombra **una** medida concreta que propondrías antes de desplegar este modelo.

*(doble clic y responde)*

> ### 🎓 Pauta docente — TODO 18 y cierre ético
>
> **Cifras:** ~20 % nocturnas, ~21 % con lluvia, ~2 % ciclistas. La intersección
> *noche + difícil* es pequeña, y encima es justo donde más faltan las velocidades (bloque 3).
>
> **Las tres respuestas esperadas:**
> 1. La conducción nocturna con mala visibilidad, y en particular las detecciones difíciles en
>    esas condiciones. Es doble carencia: hay pocos datos **y** los que hay están incompletos.
> 2. Ciclistas y peatones: los usuarios más vulnerables de la vía. Un falso negativo con un
>    vehículo es un roce; con un peatón, no.
> 3. Se acepta cualquier medida concreta y accionable: recolectar más datos nocturnos y de
>    lluvia; **evaluar el modelo por subgrupo** (métricas separadas para noche/día,
>    ciclista/vehículo) en vez de una métrica global; ponderar las clases minoritarias;
>    restringir la operación a condiciones validadas hasta tener evidencia.
>
> **La idea que debe quedar:** un promedio global oculta a las minorías. Un modelo con 97 % de
> exactitud puede tener 60 % en ciclistas nocturnos, y ese 3 % de error no está repartido al
> azar: recae sobre quienes ya son más vulnerables. Esto conecta directamente con el RA1
> ("tratamiento responsable de la información") y se retoma en la Actividad 2.2 con las métricas por clase.
>
> **Conexión con la realidad, si hay tiempo:** vale la pena mencionar que los sistemas de
> conducción autónoma reales se validan por condición operacional (ODD, *operational design
> domain*) precisamente por esto.

---
# 📝 Entrega: mini-informe

Completa la celda siguiente. Es lo que entregas al final de la sesión.

**Reglas:**
- Cada hallazgo debe incluir **una cifra**. "Hay datos sucios" no es un hallazgo; "el 3 % de
  `num_lidar_points` usa `-1` como nulo oculto" sí lo es.
- Cada decisión debe incluir **una justificación**, no solo qué hiciste.

## Informe de calidad de datos — EA1

**Estudiante:**
**Fecha:**

### Contexto
*(¿qué problema se quiere resolver y qué representa una fila del dataset?)*

### 5 hallazgos

| # | Hallazgo | Cifra | Impacto en el modelo |
|---|---|---|---|
| 1 | | | |
| 2 | | | |
| 3 | | | |
| 4 | | | |
| 5 | | | |

### 3 decisiones de preprocesamiento

| # | Decisión | Justificación | Qué se pierde |
|---|---|---|---|
| 1 | | | |
| 2 | | | |
| 3 | | | |

### 1 riesgo ético o de sesgo


### Conclusión
*(en 3 líneas: ¿está este dataset listo para entrenar un modelo? ¿qué falta?)*

---
### ✅ Antes de cerrar

- [ ] Todas las celdas ejecutan sin error (Kernel → Restart & Run All).
- [ ] Los autochequeos muestran ✅.
- [ ] El mini-informe está completo, con cifras.
- [ ] Las celdas de discusión tienen tu respuesta escrita.

### Lo que viene

- **Actividad 1.4:** impacto ético, sesgos y privacidad sobre estos mismos datos.
- **Actividad 2.2 (RA2):** aprendizaje supervisado. Ahí veremos por qué ese 2 % de
  ciclistas es un problema serio, y por qué la exactitud engaña.
- **Actividad 2.3 (RA2):** aprendizaje no supervisado — segmentación y reducción de dimensionalidad.
- **RA3:** hiperparámetros, ensamble y validación cruzada. No es "la unidad de no supervisado".

### ¿Quieres hacerlo con datos reales?

El notebook `00_opcional_waymo_real.ipynb` explica cómo bajar un fragmento del Waymo Open
Dataset real y correr **este mismo EDA** sobre él. El esquema es el mismo; el código, casi
idéntico. El ML completo (supervisado, agrupamiento, hiperparámetros) sobre varios segmentos
es `kedro run --pipeline waymo_real`. Los otros buckets de la consola (Motion, E2E cámara,
Perception v1) se **listan** en `14_opcional_waymo_buckets.ipynb`: un archivo chico, nunca el
dataset entero; no tienen modelo en este curso.

> ### 🎓 Criterios de logro de la sesión (resumen)
>
> | Nivel | Descripción |
> |---|---|
> | **Logrado** | Cuantifica el desbalance, el clima constante y la rareza de `LEVEL_2`; distingue outlier estadístico de valor de dominio; justifica cada decisión de preprocesamiento aunque el Open Dataset llegue curado. |
> | **En desarrollo** | Describe el esquema y unas pocas cifras, pero no conecta desbalance / sesgo de muestreo con el modelado. |
> | **Inicial** | Ejecuta el notebook sin interpretar; el informe no tiene cifras. |
>
> **Hallazgos medidos en el lote de 40 segmentos (2026-09-08):**
>
> 1. 530.396 filas × 16 columnas, 40 `segment_id`, 0 % nulos
> 2. `object_type`: vehicle 48,43 % · sign 26,46 % · pedestrian 24,67 % · cyclist **0,45 %**
> 3. `detection_difficulty`: LEVEL_1 87,67 % · LEVEL_2 **12,33 %**
> 4. `weather`: **100 % `sunny`** (sesgo del censo, no de la Tierra)
> 5. `location`: SF 398.065 · PHX 132.331
> 6. `time_of_day`: Day 461.090 · Night 51.867 · Dawn/Dusk 17.439
> 7. Mediana `speed_mps` **0,0133** (muchos objetos parados o casi)
> 8. Valores imposibles: **0**
> 9. Un solo segmento no alcanza para train/test sin fuga
> 10. `camera_box` y E2E **no** entran al modelo
>
> **Señal de alerta durante la clase:** si un grupo termina diciendo que hay que `dropna()`,
> no miró el 0 % de nulos. Pregúntale el % de `cyclist` y el de `sunny`.